# 00 - Environment Setup & Sanity Checks
## XAI Skin Lesion Classification - Research Pipeline

**Purpose**: Verify your environment is ready before running any research experiments.

---

## CELL 1: Environment Sanity Check

**WHY**: Catch environment problems before wasting hours training.

Run this first on any new machine or Python environment.

In [ ]:
import torch
import torchvision
import timm
import numpy as np
import pandas as pd
import matplotlib
import sklearn
import cv2
import albumentations

print("=== Environment Check ===")
print(f"PyTorch:     {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")
print(f"timm:        {timm.__version__}")
print(f"NumPy:       {np.__version__}")
print(f"OpenCV:      {cv2.__version__}")
print(f"Scikit-learn:{sklearn.__version__}")

print(f"\n=== GPU ===")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    # Quick compute test
    a = torch.randn(1000, 1000).cuda()
    b = torch.randn(1000, 1000).cuda()
    c = a @ b
    print(f"Matrix multiply test: PASSED")
else:
    print("WARNING: No GPU — training will be VERY slow. Consider Google Colab.")

---

## CELL 2: Data Sanity Check

**WHY**: Confirm data loaded correctly before training.

**Prerequisite**: You must have downloaded the HAM10000 dataset and run the data pipeline.
The backend repo at `../Skin_Lesion_Classification_backend/` should contain:
- `ml/data/processed/metadata_with_paths.csv` - generated metadata
- `ml/data/raw/` - original HAM10000 images

In [ ]:
import pandas as pd
from pathlib import Path

# Point to backend data directory
DATA_DIR = Path("../Skin_Lesion_Classification_backend/ml/data/processed")

df = pd.read_csv(DATA_DIR / "metadata_with_paths.csv")

print("=== Dataset Overview ===")
print(f"Total images:      {len(df)}")
print(f"Unique patients:   {df['patient_id'].nunique()}")
print(f"Unique lesion types: {df['dx'].nunique()}")
print(f"\nClass distribution:")
print(df['dx'].value_counts())
print(f"\nBinary label distribution:")
print(df['label_name'].value_counts())
print(f"\nImbalance ratio: {df['label'].value_counts()[0] / df['label'].value_counts()[1]:.2f}:1 (benign:malignant)")

# Check for missing files
missing = df[df['filepath'].isna()]
print(f"\nMissing image files: {len(missing)}")

# Check image is actually readable
import random
from PIL import Image
sample_paths = df['filepath'].dropna().sample(5, random_state=42).tolist()
for p in sample_paths:
    try:
        img = Image.open(p)
        print(f"OK: {Path(p).name} — {img.size} {img.mode}")
    except Exception as e:
        print(f"BROKEN: {p} — {e}")

---

## CELL 3: Data Split Leakage Check

**WHY**: Patient leakage is the #1 mistake in medical ML papers.
If the same patient appears in train AND test, your results are artificially inflated.
This test proves you have no leakage.

**Expected result**: All overlap counts must be 0.

In [ ]:
import sys
sys.path.append("../Skin_Lesion_Classification_backend/ml")
from src.data.dataset import create_splits

train_df, val_df, test_df = create_splits(df)

print("=== Split Sizes ===")
print(f"Train: {len(train_df)} images, {train_df['patient_id'].nunique()} patients")
print(f"Val:   {len(val_df)} images,   {val_df['patient_id'].nunique()} patients")
print(f"Test:  {len(test_df)} images,  {test_df['patient_id'].nunique()} patients")

# THE CRITICAL TEST — must all be 0
train_patients = set(train_df['patient_id'])
val_patients   = set(val_df['patient_id'])
test_patients  = set(test_df['patient_id'])

overlap_tv = train_patients & val_patients
overlap_tt = train_patients & test_patients
overlap_vt = val_patients & test_patients

print(f"\n=== Leakage Check ===")
print(f"Train/Val overlap:  {len(overlap_tv)} patients  ← must be 0")
print(f"Train/Test overlap: {len(overlap_tt)} patients  ← must be 0")
print(f"Val/Test overlap:   {len(overlap_vt)} patients  ← must be 0")

if len(overlap_tv) == 0 and len(overlap_tt) == 0 and len(overlap_vt) == 0:
    print("\n✅ No data leakage. Split is valid.")
else:
    print("\n❌ LEAKAGE DETECTED. Fix create_splits() before training!")

# Check class balance in each split
for name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    ratio = split_df['label'].mean()
    print(f"{name} malignant ratio: {ratio:.3f} ({ratio*100:.1f}%)")

---

## CELL 4: DataLoader Sanity Check

**WHY**: Confirm transforms work, tensor shapes are correct,
and WeightedRandomSampler is actually balancing the classes.

In [ ]:
from src.data.dataset import get_dataloaders
import matplotlib.pyplot as plt
import numpy as np

train_loader, val_loader, test_loader, splits = get_dataloaders(
    df, batch_size=8, img_size=224, num_workers=0
)

# Get one batch
images, labels, ids = next(iter(train_loader))
print(f"Image tensor shape: {images.shape}")   # (8, 3, 224, 224)
print(f"Label tensor shape: {labels.shape}")   # (8,)
print(f"Label values:       {labels.tolist()}")
print(f"Value range:        [{images.min():.3f}, {images.max():.3f}]")  # Should be approx [-2, 2] after normalize

# Visualize — ALWAYS look at your data!
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
# De-normalize for display (ImageNet mean/std)
mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

for i in range(8):
    ax = axes[i // 4][i % 4]
    img = images[i].permute(1, 2, 0).numpy()
    img = img * std + mean  # de-normalize
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    ax.set_title(f"{'Malignant' if labels[i] == 1 else 'Benign'}", 
                 color='red' if labels[i] == 1 else 'green', fontweight='bold')
    ax.axis('off')

plt.suptitle('Training Batch (verify augmentations look reasonable)', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/figures/sanity_training_batch.png', dpi=150)
plt.show()

# Check WeightedSampler is working by counting labels over many batches
print("\n=== Sampler Balance Check (should be ~50/50) ===")
label_counts = {0: 0, 1: 0}
for _, labels_batch, _ in train_loader:
    for l in labels_batch.tolist():
        label_counts[l] += 1
total = sum(label_counts.values())
print(f"Benign:    {label_counts[0]} ({label_counts[0]/total*100:.1f}%)")
print(f"Malignant: {label_counts[1]} ({label_counts[1]/total*100:.1f}%)")
print("If benign >> malignant, your sampler is broken!")

---

## CELL 5: Verify Model Checkpoints Exist

**WHY**: Most research notebooks require trained model weights.
Check that they exist before running experiments.

In [ ]:
from pathlib import Path

MODEL_DIR = Path("../Skin_Lesion_Classification_backend/ml/outputs/models")
CKPT_DIR  = Path("../Skin_Lesion_Classification_backend/ml/outputs/models/checkpoints")

print("=== Model Checkpoints ===")
for p in sorted(MODEL_DIR.glob("*.pth")):
    size_mb = p.stat().st_size / 1e6
    print(f"  {p.name} ({size_mb:.1f} MB)")

print("\n=== Training Checkpoints (for RQ5) ===")
for p in sorted(CKPT_DIR.glob("*.pth")):
    print(f"  {p.name}")

if not list(MODEL_DIR.glob("*.pth")):
    print("\n⚠️ No model checkpoints found. Train models before running RQ notebooks.")
    print("    See DL_SCIENTIST_TESTING_GUIDE.md in the backend repo for training instructions.")